In [ ]:
# P6 TRAINING DATA CLEANUP — Phase 1: Automated Regex Fixes

import json
import re
import os
from collections import Counter

# Mount Google Drive & Set Paths
from google.colab import drive
drive.mount('/content/drive')

# CONFIG
# Set input/output paths
BASE_DIR = "/content/drive/MyDrive/UPGRAD_MSML_C26"
INPUT_FILE = f"{BASE_DIR}/training_data_v6/whisper_P5.1C_READY_TO_TRAIN_clean.json"
OUTPUT_FILE = f"{BASE_DIR}/training_data_v6/whisper_P6_CLEAN-PH1.json"

with open(INPUT_FILE) as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples from P5.1C")
print("=" * 70)

# Track all changes for audit trail
change_log = []

def log_change(index, fix_name, before, after):
    """Record every change for review."""
    if before != after:
        change_log.append({
            "index": index,
            "fix": fix_name,
            "before": before[:100],
            "after": after[:100]
        })

# FIX 1: POSSESSIVE PRICE GRAMMAR
# "Scroll's 250 points" → "A scroll costs 250 points"
#   "scroll's 212 points" (discounted prices)


def fix_possessive_price(response: str) -> str:
    """Fix possessive price grammar while preserving non-price possessives."""
    fixed = response

    # Pattern: "Item's <NUMBER> points" → "A item costs <NUMBER> points"
    # Matches: Scroll's 250, scroll's 212, Hint's 150, hint's 127
    # Does NOT match: Scroll's your best, Hint's gonna help

    # Scroll's <number>
    fixed = re.sub(
        r"\bScroll's\s+(\d+)\s+(points?)",
        r"A scroll costs \1 \2",
        fixed
    )
    fixed = re.sub(
        r"\bscroll's\s+(\d+)\s+(points?)",
        r"a scroll costs \1 \2",
        fixed
    )

    # Hint's <number>
    fixed = re.sub(
        r"\bHint's\s+(\d+)\s+(points?)",
        r"A hint costs \1 \2",
        fixed
    )
    fixed = re.sub(
        r"\bhint's\s+(\d+)\s+(points?)",
        r"a hint costs \1 \2",
        fixed
    )

    # Solution's <number>
    fixed = re.sub(
        r"\bSolution's\s+(\d+)\s+(points?)",
        r"A solution costs \1 \2",
        fixed
    )
    fixed = re.sub(
        r"\bsolution's\s+(\d+)\s+(points?)",
        r"a solution costs \1 \2",
        fixed
    )

    # "Scroll's 250 and helps you" → "A scroll is 250 points and helps you"
    fixed = re.sub(
        r"\b([Ss])croll's\s+(\d{3})(?=\s+(?:and|but|—|,|for|if|which|that|to))",
        lambda m: f"{'A' if m.group(1) == 'S' else 'a'} scroll is {m.group(2)} points",
        fixed
    )
    fixed = re.sub(
        r"\b([Hh])int's\s+(\d{2,3})(?=\s+(?:and|but|—|,|for|if|which|that|to))",
        lambda m: f"{'A' if m.group(1) == 'H' else 'a'} hint is {m.group(2)} points",
        fixed
    )
    fixed = re.sub(
        r"\b([Ss])olution's\s+(\d{3})(?=\s+(?:and|but|—|,|for|if|which|that|to))",
        lambda m: f"{'A' if m.group(1) == 'S' else 'a'} solution is {m.group(2)} points",
        fixed
    )

    # Catch: "Hint's 150." at end of sentence
    fixed = re.sub(
        r"\b([Ss])croll's\s+(\d{3})\.",
        lambda m: f"{'A' if m.group(1) == 'S' else 'a'} scroll costs {m.group(2)} points.",
        fixed
    )
    fixed = re.sub(
        r"\b([Hh])int's\s+(\d{2,3})\.",
        lambda m: f"{'A' if m.group(1) == 'H' else 'a'} hint costs {m.group(2)} points.",
        fixed
    )
    fixed = re.sub(
        r"\b([Ss])olution's\s+(\d{3})\.",
        lambda m: f"{'A' if m.group(1) == 'S' else 'a'} solution costs {m.group(2)} points.",
        fixed
    )

    return fixed

# FIX 2: POSSESSIVE COMPARISON TEMPLATES
# "Hint's 105 points for you. But scroll's 175 and works every time."
# → "A hint costs 105 points. But a scroll at 175 works every time."

def fix_comparison_template(response: str) -> str:
    """Fix remaining comparison patterns after possessive fix."""
    fixed = response

    # "the scroll at 250 points" is fine — canonical form
    # "scroll at 250" is fine
    # We just need to catch any remaining possessive comparisons
    # that Fix 1 might have missed

    # Catch: "a scroll's your best bet" (non-price possessive)
    # These are OK — "scroll's" here means "scroll is" not "scroll costs"
    # So we leave these alone.

    # Catch: "scroll for 250 points" → already fine
    # Catch: "250 points for a scroll" → already fine

    return fixed

# FIX 3: BANNED PHRASES

def fix_banned_phrases(response: str) -> str:
    """Replace V6 banned phrases with approved alternatives."""
    fixed = response

    # ----- "shall" (542 instances) -----
    # Context-aware replacements for natural flow
    fixed = re.sub(r"\bshall be yours\b", "is yours", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall I\b", "want me to", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall we\b", "let's", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall guide\b", "will guide", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall help\b", "will help", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall serve\b", "will serve", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall protect\b", "will protect", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall see\b", "will see", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall light\b", "will light", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall keep\b", "will keep", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall make\b", "will make", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall give\b", "will give", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall have\b", "will have", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall find\b", "will find", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall lead\b", "will lead", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bit shall\b", "it will", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bthat shall\b", "that will", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bwhich shall\b", "which will", fixed, flags=re.IGNORECASE)
    # Generic fallback for any remaining "shall"
    fixed = re.sub(r"\bshall\b", "will", fixed, flags=re.IGNORECASE)

    # ----- "traveler" (104 instances) -----
    fixed = re.sub(r"\bDear traveler\b", "Hey friend", fixed)
    fixed = re.sub(r"\bdear traveler\b", "hey friend", fixed)
    fixed = re.sub(r"\bTraveler\b", "Friend", fixed)
    fixed = re.sub(r"\btraveler\b", "friend", fixed)
    fixed = re.sub(r"\btravelers\b", "friends", fixed, flags=re.IGNORECASE)

    # ----- "illuminate" (21 instances) -----
    fixed = re.sub(r"\billuminate your path\b", "light your way", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\billuminate the\b", "light up the", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\billuminate\b", "light up", fixed, flags=re.IGNORECASE)

    # ----- "coffers" (1 instance) -----
    fixed = re.sub(r"\byour coffers\b", "your stash", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bcoffers\b", "stash", fixed, flags=re.IGNORECASE)

    # ----- "wanderer" (if present) -----
    fixed = re.sub(r"\bwanderer\b", "friend", fixed, flags=re.IGNORECASE)

    return fixed

# FIX 4: CLEANUP ARTIFACTS

def fix_artifacts(response: str) -> str:
    """Clean up artifacts introduced by regex fixes."""
    fixed = response
    # Double spaces
    fixed = re.sub(r"  +", " ", fixed)
    # "A a scroll" from double application
    fixed = re.sub(r"\bA a scroll\b", "A scroll", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bA a hint\b", "A hint", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bA a solution\b", "A solution", fixed, flags=re.IGNORECASE)
    # Trim
    fixed = fixed.strip()
    return fixed


# APPLY ALL FIXES

print("\nApplying P6 fixes...")
print("-" * 70)

fix_counts = Counter()

for sample in data:
    idx = sample.get('index', '?')
    original = sample['whisper_response']

    # Apply fixes in order
    step1 = fix_possessive_price(original)
    if step1 != original:
        fix_counts['possessive_price'] += 1
        log_change(idx, 'possessive_price', original, step1)

    step2 = fix_comparison_template(step1)
    if step2 != step1:
        fix_counts['comparison_template'] += 1
        log_change(idx, 'comparison_template', step1, step2)

    step3 = fix_banned_phrases(step2)
    if step3 != step2:
        # Count individual phrase fixes
        if re.search(r'\bshall\b', original, re.I) and not re.search(r'\bshall\b', step3, re.I):
            fix_counts['shall'] += 1
        if re.search(r'\btraveler\b', original, re.I) and not re.search(r'\btraveler\b', step3, re.I):
            fix_counts['traveler'] += 1
        if re.search(r'\billuminate\b', original, re.I) and not re.search(r'\billuminate\b', step3, re.I):
            fix_counts['illuminate'] += 1
        if re.search(r'\bcoffers\b', original, re.I) and not re.search(r'\bcoffers\b', step3, re.I):
            fix_counts['coffers'] += 1
        log_change(idx, 'banned_phrases', step2, step3)

    step4 = fix_artifacts(step3)
    if step4 != step3:
        fix_counts['artifacts'] += 1
        log_change(idx, 'artifacts', step3, step4)

    # Count total changed
    if step4 != original:
        fix_counts['total_changed'] += 1

    sample['whisper_response'] = step4

# SUMMARY

print(f"\n{'=' * 70}")
print(f"P6 FIX SUMMARY")
print(f"{'=' * 70}")
print(f"  Total samples: {len(data)}")
print(f"  Samples modified: {fix_counts['total_changed']}")
print(f"  Unchanged: {len(data) - fix_counts['total_changed']}")
print(f"\n  Fix breakdown:")
print(f"    Possessive price grammar: {fix_counts['possessive_price']}")
print(f"    Comparison templates:     {fix_counts['comparison_template']}")
print(f"    'shall' removed:          {fix_counts['shall']}")
print(f"    'traveler' removed:       {fix_counts['traveler']}")
print(f"    'illuminate' removed:     {fix_counts['illuminate']}")
print(f"    'coffers' removed:        {fix_counts['coffers']}")
print(f"    Artifact cleanup:         {fix_counts['artifacts']}")


# VERIFY: Re-run audit checks on cleaned data

print(f"\n{'=' * 70}")
print(f"POST-FIX VERIFICATION")
print(f"{'=' * 70}")

# Check residuals
residual_possessive = sum(1 for s in data
    if re.search(r"(?:scroll|hint|solution)'s\s+\d+", s['whisper_response'], re.I))
residual_shall = sum(1 for s in data
    if re.search(r'\bshall\b', s['whisper_response'], re.I))
residual_traveler = sum(1 for s in data
    if re.search(r'\btraveler\b', s['whisper_response'], re.I))
residual_illuminate = sum(1 for s in data
    if re.search(r'\billuminate\b', s['whisper_response'], re.I))

print(f"  Residual possessive price: {residual_possessive} (target: 0)")
print(f"  Residual 'shall': {residual_shall} (target: 0)")
print(f"  Residual 'traveler': {residual_traveler} (target: 0)")
print(f"  Residual 'illuminate': {residual_illuminate} (target: 0)")

status = "ALL CLEAN" if (residual_possessive + residual_shall + residual_traveler + residual_illuminate) == 0 else "⚠️ RESIDUALS FOUND"
print(f"\n  Status: {status}")

# few examples of before/after
print(f"\n{'=' * 70}")
print(f"SAMPLE CHANGES (first 10)")
print(f"{'=' * 70}")
for change in change_log[:10]:
    print(f"\n  [{change['index']}] {change['fix']}")
    print(f"    Before: {change['before']}...")
    print(f"    After:  {change['after']}...")

#Save

with open(OUTPUT_FILE, 'w') as f:
    json.dump(data, f, indent=2)

print(f"\n{'=' * 70}")
print(f"✅ SAVED: {OUTPUT_FILE}")
print(f"   Samples: {len(data)}")
print(f"   Size: {os.path.getsize(OUTPUT_FILE) / 1e6:.1f} MB")
print(f"{'=' * 70}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 3110 samples from P5.1C

Applying P6 fixes...
----------------------------------------------------------------------

P6 FIX SUMMARY
  Total samples: 3110
  Samples modified: 943
  Unchanged: 2167

  Fix breakdown:
    Possessive price grammar: 381
    Comparison templates:     0
    'shall' removed:          542
    'traveler' removed:       104
    'illuminate' removed:     21
    'coffers' removed:        1
    Artifact cleanup:         49

POST-FIX VERIFICATION
  Residual possessive price: 5 (target: 0)
  Residual 'shall': 0 (target: 0)
  Residual 'traveler': 0 (target: 0)
  Residual 'illuminate': 0 (target: 0)

  Status: ⚠️ RESIDUALS FOUND

SAMPLE CHANGES (first 10)

  [0] possessive_price
    Before: Nice call. Hint's 150 points — should give you a solid nudge on this puzzle. Let's see where it lead...
    After:  Nice call. A hint costs 150 poin

In [ ]:
# P6 Phase 1.5: Fix Conversation History in full_context

import json
import re
import os
from collections import Counter

# ===================== CONFIG =====================
BASE_DIR = "/content/drive/MyDrive/UPGRAD_MSML_C26"
INPUT_FILE = f"{BASE_DIR}/training_data_v6/whisper_P6_CLEAN-PH1.json"
OUTPUT_FILE = f"{BASE_DIR}/training_data_v6/whisper_P6_CLEAN.json"
# ==================================================

with open(INPUT_FILE) as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples")
print("=" * 70)
print("PHASE 1.5: Fixing embedded Whisper responses in [CONVERSATION HISTORY]")
print("=" * 70)

# ---- Reuse the same fix functions from Phase 1 ----

def fix_possessive_price(text: str) -> str:
    """Fix possessive price grammar."""
    fixed = text
    # Scroll's <number> points
    fixed = re.sub(r"\bScroll's\s+(\d+)\s+(points?)", r"A scroll costs \1 \2", fixed)
    fixed = re.sub(r"\bscroll's\s+(\d+)\s+(points?)", r"a scroll costs \1 \2", fixed)
    # Hint's <number> points
    fixed = re.sub(r"\bHint's\s+(\d+)\s+(points?)", r"A hint costs \1 \2", fixed)
    fixed = re.sub(r"\bhint's\s+(\d+)\s+(points?)", r"a hint costs \1 \2", fixed)
    # Solution's <number> points
    fixed = re.sub(r"\bSolution's\s+(\d+)\s+(points?)", r"A solution costs \1 \2", fixed)
    fixed = re.sub(r"\bsolution's\s+(\d+)\s+(points?)", r"a solution costs \1 \2", fixed)
    # Without "points" suffix
    fixed = re.sub(
        r"\b([Ss])croll's\s+(\d{3})(?=\s+(?:and|but|—|,|for|if|which|that|to))",
        lambda m: f"{'A' if m.group(1) == 'S' else 'a'} scroll is {m.group(2)} points",
        fixed
    )
    fixed = re.sub(
        r"\b([Hh])int's\s+(\d{2,3})(?=\s+(?:and|but|—|,|for|if|which|that|to))",
        lambda m: f"{'A' if m.group(1) == 'H' else 'a'} hint is {m.group(2)} points",
        fixed
    )
    fixed = re.sub(
        r"\b([Ss])olution's\s+(\d{3})(?=\s+(?:and|but|—|,|for|if|which|that|to))",
        lambda m: f"{'A' if m.group(1) == 'S' else 'a'} solution is {m.group(2)} points",
        fixed
    )
    # End of sentence
    fixed = re.sub(
        r"\b([Ss])croll's\s+(\d{3})\.",
        lambda m: f"{'A' if m.group(1) == 'S' else 'a'} scroll costs {m.group(2)} points.",
        fixed
    )
    fixed = re.sub(
        r"\b([Hh])int's\s+(\d{2,3})\.",
        lambda m: f"{'A' if m.group(1) == 'H' else 'a'} hint costs {m.group(2)} points.",
        fixed
    )
    fixed = re.sub(
        r"\b([Ss])olution's\s+(\d{3})\.",
        lambda m: f"{'A' if m.group(1) == 'S' else 'a'} solution costs {m.group(2)} points.",
        fixed
    )
    return fixed


def fix_banned_phrases(text: str) -> str:
    """Replace V6 banned phrases."""
    fixed = text
    fixed = re.sub(r"\bshall be yours\b", "is yours", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall I\b", "want me to", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall we\b", "let's", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bshall\b", "will", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bDear traveler\b", "Hey friend", fixed)
    fixed = re.sub(r"\bdear traveler\b", "hey friend", fixed)
    fixed = re.sub(r"\bTraveler\b", "Friend", fixed)
    fixed = re.sub(r"\btraveler\b", "friend", fixed)
    fixed = re.sub(r"\btravelers\b", "friends", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\billuminate your path\b", "light your way", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\billuminate\b", "light up", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\byour coffers\b", "your stash", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bcoffers\b", "stash", fixed, flags=re.IGNORECASE)
    fixed = re.sub(r"\bwanderer\b", "friend", fixed, flags=re.IGNORECASE)
    return fixed


# ---- Identity patterns to detect in history ----
IDENTITY_QUESTIONS = re.compile(
    r"Player:\s*(who are you|what are you|what's your name|what is your name|"
    r"tell me about yourself|who is this|identify yourself)",
    re.IGNORECASE
)

# Pool of short identity responses for history injection
HISTORY_IDENTITY_RESPONSES = [
    "I'm Whisper. The merchant at these gates.",
    "Name's Whisper. I sell hints and scrolls around here.",
    "Call me Whisper, friend. I'm the merchant.",
    "Whisper. That's me. Just a merchant doing my thing.",
    "I'm Whisper — the chill merchant at these gates.",
    "Name's Whisper. Been running this shop a while now.",
    "They call me Whisper. I deal in hints and scrolls.",
    "Whisper, friend. Your local gate merchant.",
    "I go by Whisper. Just a merchant who knows these gates well.",
    "I'm Whisper. Merchant, mystery, all that good stuff.",
]

import random
random.seed(42)


def fix_conversation_history(full_context: str) -> str:
    """
    Fix all Whisper responses embedded in [CONVERSATION HISTORY].
    Applies possessive price, banned phrase, and identity fixes.
    """
    # Only process if there's a conversation history section
    if '[CONVERSATION HISTORY]' not in full_context:
        return full_context

    # Split context at conversation history
    parts = full_context.split('[CONVERSATION HISTORY]')
    if len(parts) != 2:
        return full_context

    before_history = parts[0]
    history_and_after = parts[1]

    # Split history from the next section (likely [RL DECISION])
    # Find the next section header
    next_section = re.search(r'\n\[(?:RL DECISION|EFFECTIVE PRICES|PLAYER STATE|GAME STATE)\]', history_and_after)
    if next_section:
        history_text = history_and_after[:next_section.start()]
        after_history = history_and_after[next_section.start():]
    else:
        history_text = history_and_after
        after_history = ""

    # Process each Whisper line in history
    lines = history_text.split('\n')
    fixed_lines = []
    prev_was_identity_question = False

    for line in lines:
        if line.startswith('Whisper:'):
            whisper_text = line[len('Whisper:'):].strip()

            # Apply standard fixes
            whisper_text = fix_possessive_price(whisper_text)
            whisper_text = fix_banned_phrases(whisper_text)

            # Fix identity responses that don't mention Whisper
            if prev_was_identity_question and not re.search(r'whisper', whisper_text, re.I):
                whisper_text = random.choice(HISTORY_IDENTITY_RESPONSES)

            fixed_lines.append(f"Whisper: {whisper_text}")
            prev_was_identity_question = False
        else:
            # Check if this is an identity question
            if IDENTITY_QUESTIONS.search(line):
                prev_was_identity_question = True
            else:
                prev_was_identity_question = False
            fixed_lines.append(line)

    # Reconstruct
    fixed_history = '\n'.join(fixed_lines)
    return before_history + '[CONVERSATION HISTORY]' + fixed_history + after_history

# APPLY TO ALL SAMPLES

multi_turn_count = 0
history_fixed_count = 0

for sample in data:
    ctx = sample.get('full_context', '')
    if '[CONVERSATION HISTORY]' in ctx and '(No previous conversation)' not in ctx:
        multi_turn_count += 1
        original = ctx
        fixed = fix_conversation_history(ctx)
        if fixed != original:
            sample['full_context'] = fixed
            history_fixed_count += 1

print(f"\n  Multi-turn samples with history: {multi_turn_count}")
print(f"  Samples with history fixes applied: {history_fixed_count}")

# VERIFY

print(f"\n{'=' * 70}")
print("VERIFICATION")
print("=" * 70)

# Check for residual issues in full_context
residual_shall_ctx = 0
residual_traveler_ctx = 0
residual_possessive_ctx = 0
identity_no_name_ctx = 0

for sample in data:
    ctx = sample.get('full_context', '')
    # Only check within [CONVERSATION HISTORY] section
    if '[CONVERSATION HISTORY]' not in ctx or '(No previous conversation)' in ctx:
        continue

    history_match = re.search(r'\[CONVERSATION HISTORY\](.*?)(?:\[RL DECISION\]|\Z)', ctx, re.DOTALL)
    if not history_match:
        continue

    history = history_match.group(1)

    # Extract Whisper lines
    whisper_lines = re.findall(r'Whisper:\s*(.*)', history)
    for wl in whisper_lines:
        if re.search(r'\bshall\b', wl, re.I):
            residual_shall_ctx += 1
        if re.search(r'\btraveler\b', wl, re.I):
            residual_traveler_ctx += 1
        if re.search(r"(?:scroll|hint|solution)'s\s+\d+", wl, re.I):
            residual_possessive_ctx += 1

print(f"  Residual 'shall' in history: {residual_shall_ctx}")
print(f"  Residual 'traveler' in history: {residual_traveler_ctx}")
print(f"  Residual possessive price in history: {residual_possessive_ctx}")

# Sample before/after
print(f"\n  Sample fixes (first 3 multi-turn with changes):")
shown = 0
for sample in data:
    if shown >= 3:
        break
    ctx = sample.get('full_context', '')
    if '[CONVERSATION HISTORY]' in ctx and 'Whisper:' in ctx and '(No previous conversation)' not in ctx:
        history_match = re.search(r'\[CONVERSATION HISTORY\](.*?)(?:\[RL DECISION\]|\Z)', ctx, re.DOTALL)
        if history_match:
            history = history_match.group(1).strip()
            print(f"\n    [{sample.get('index','?')}] action={sample['action']}")
            # Show first Whisper line
            whisper_lines = re.findall(r'Whisper:\s*(.*)', history)
            if whisper_lines:
                print(f"    History Whisper: \"{whisper_lines[0][:80]}...\"")
            shown += 1

# Save


with open(OUTPUT_FILE, 'w') as f:
    json.dump(data, f, indent=2)

print(f"✅ SAVED: {OUTPUT_FILE} (overwritten with history fixes)")
print(f"   Samples: {len(data)}")

Loaded 3110 samples
PHASE 1.5: Fixing embedded Whisper responses in [CONVERSATION HISTORY]

  Multi-turn samples with history: 3110
  Samples with history fixes applied: 79

VERIFICATION
  Residual 'shall' in history: 0
  Residual 'traveler' in history: 0
  Residual possessive price in history: 0

  Sample fixes (first 3 multi-turn with changes):

    [0] action=standard_offer
    History Whisper: "Hints are whispers from the wind, guiding you through the shadows. They show sec..."

    [5] action=standard_offer
    History Whisper: "Scrolls hold whispers of ancient power. They can unlock paths yet unseen.  A mer..."

    [9] action=approve_loan
    History Whisper: "I sense your urgency. The path you tread is not without its shadows. What weighs..."
✅ SAVED: /content/drive/MyDrive/UPGRAD_MSML_C26/training_data_v6/whisper_P6_CLEAN.json (overwritten with history fixes)
   Samples: 3110


In [ ]:
# P6 TRAINING DATA CLEANUP — Phase 2: Identity Rewrites + Off-Topic Samples

import json
import re
import os
import random
from copy import deepcopy

random.seed(42)

# CONFIG
BASE_DIR = "/content/drive/MyDrive/UPGRAD_MSML_C26"
INPUT_FILE = f"{BASE_DIR}/training_data_v6/whisper_P6_CLEAN.json"
OUTPUT_FILE = f"{BASE_DIR}/training_data_v6/whisper_P6_FINAL.json"

with open(INPUT_FILE) as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples from P6_CLEAN")
print("=" * 70)

# PHASE 2A: IDENTITY SAMPLE REWRITES

print("\n" + "=" * 70)
print("PHASE 2A: IDENTITY SAMPLE REWRITES")
print("=" * 70)

# Pool of identity response templates
# Each includes "Whisper" naturally + stays in character
# Grouped by style to ensure diversity

WHISPER_IDENTITY_TEMPLATES = [
    # Style 1: Direct name + role (30%)
    "Name's Whisper. I'm the merchant around here — got hints, scrolls, the works.",
    "I'm Whisper. Just your local merchant, hanging by the gates.",
    "They call me Whisper. I sell hints, scrolls, and a bit of wisdom on the side.",
    "Whisper. That's what they call me around here. I run this little shop by the gates.",
    "I'm Whisper, the merchant. Been set up here for a while now.",
    "Name's Whisper, friend. I deal in hints, scrolls, and the occasional NFT.",
    "I'm Whisper — the chill merchant at the Gate of Whispers. Need anything?",
    "Call me Whisper. I'm the one who keeps the shop running by these gates.",
    "Whisper's the name. I sell what you need to survive these gates.",
    "I'm Whisper. Merchant, advisor, sometimes storyteller. Depends on the day.",
    "Name's Whisper. Been running this shop since... well, a while now.",
    "I go by Whisper. Just a merchant who knows a thing or two about these gates.",
    "They call me Whisper. Been trading hints and scrolls here for a good while.",
    "I'm Whisper, friend. The merchant at the gates. What can I do for you?",
    "Whisper — that's me. Just a guy who sells useful stuff around here.",

    # Style 2: Mystery + name (25%)
    "Whisper. Just Whisper. Where I came from? That's a story for another time.",
    "Name's Whisper. How I got here is a long story — the short version is, I sell stuff.",
    "Call me Whisper. The gates have their secrets, and so do I.",
    "I'm Whisper. Showed up at these gates a while back. Nobody really knows from where.",
    "Whisper. That's all you need to know for now. But if you need help, I got you.",
    "Name's Whisper. I don't talk much about my past, but I know these gates well.",
    "They call me Whisper — fitting name for someone who deals in secrets and shadows.",
    "I'm Whisper. Three winters at these gates, and still nobody knows my full story.",
    "Call me Whisper. Some things are better left mysterious, you know?",
    "Whisper. The name fits — I hear things, I know things. And I sell things.",
    "I'm Whisper. Where I'm from doesn't matter much. What matters is I can help.",
    "Name's Whisper, and that's about all I'll give you for free.",

    # Style 3: Casual + name (25%)
    "Yo, I'm Whisper. The chill merchant at these gates. What's good?",
    "Whisper, friend. Just vibing here at the gates, selling hints and scrolls.",
    "Name's Whisper. I'm basically the guy you come to when puzzles get tough.",
    "I'm Whisper — just hanging out at the gates, helping folks like you.",
    "Call me Whisper. I'm the go-to merchant around here. Pretty simple.",
    "Whisper's what everyone calls me. I keep it chill and sell what you need.",
    "I'm Whisper, your local gate merchant. Nothing fancy, just good deals.",
    "Hey, I'm Whisper. I sell stuff, give advice, and keep things real.",
    "Name's Whisper. Just a chill dude running a shop at the gates.",
    "Whisper here. Merchant life is pretty straightforward — you need it, I got it.",
    "I'm Whisper. Think of me as your friendly neighborhood merchant.",
    "Call me Whisper, friend. I handle the trading around here.",

    # Style 4: Playful + name (20%)
    "Whisper! At your service. Well, at your service for the right price anyway.",
    "I'm Whisper. No, it's not because I whisper — it's a whole thing. Anyway, need something?",
    "Name's Whisper. Mysterious enough for you? Good. Now, want to buy something?",
    "Call me Whisper. Yeah, the name's a bit dramatic, but it works. What do you need?",
    "I'm Whisper — merchant, mystery, and occasional comedian. Mostly merchant though.",
    "Whisper. Cool name, right? I think so too. Anyway, I sell hints and scrolls.",
    "They gave me the name Whisper. I didn't pick it, but I'm not complaining.",
    "I'm Whisper! The one and only merchant at these gates. Catchy name, huh?",
    "Name's Whisper. Before you ask — no, I don't know why either. But here I am.",
    "Call me Whisper. It's got a nice ring to it. Now, what brings you here?",
]

# Find identity samples without "Whisper"
identity_samples = [(i, s) for i, s in enumerate(data)
                    if s.get('action') == 'identity_answer'
                    and not re.search(r'whisper', s['whisper_response'], re.I)]

print(f"  Identity samples missing 'Whisper': {len(identity_samples)}")
print(f"  Template pool size: {len(WHISPER_IDENTITY_TEMPLATES)}")

# Shuffle templates and cycle through them
random.shuffle(WHISPER_IDENTITY_TEMPLATES)
template_idx = 0

rewrite_count = 0
for data_idx, sample in identity_samples:
    old_response = sample['whisper_response']

    # Pick next template (cycle through all)
    new_response = WHISPER_IDENTITY_TEMPLATES[template_idx % len(WHISPER_IDENTITY_TEMPLATES)]
    template_idx += 1

    data[data_idx]['whisper_response'] = new_response
    rewrite_count += 1

print(f"  Rewrites applied: {rewrite_count}")

# Verify: check identity samples now
identity_with_name = sum(1 for s in data
    if s.get('action') == 'identity_answer'
    and re.search(r'whisper', s['whisper_response'], re.I))
total_identity = sum(1 for s in data if s.get('action') == 'identity_answer')
print(f"  Identity samples with 'Whisper': {identity_with_name}/{total_identity} ({identity_with_name/total_identity*100:.0f}%)")

# Show a few examples
print(f"\n  Sample rewrites:")
for _, sample in identity_samples[:5]:
    print(f"    [{sample['index']}] {sample['whisper_response'][:90]}...")

# PHASE 2B: OFF-TOPIC DEFLECTION SAMPLES

print(f"\n{'=' * 70}")
print("PHASE 2B: OFF-TOPIC DEFLECTION SAMPLES")
print("=" * 70)

# Template for a game context that we'll reuse for off-topic samples
# Using moderate game state — nothing extreme
def make_off_topic_context(level=3, points=450, pol=20, curses=1,
                            golden_gates=1, hints_pct=60, scrolls_pct=55):
    return f"""[GAME STATE]
Level: {level}/7 | Golden Gates: {golden_gates}/5
Stock: Hints {hints_pct}% | Scrolls {scrolls_pct}%

[PLAYER STATE]
Points: {points} | POL: {pol} | Curses: {curses}/4

[EFFECTIVE PRICES]
Hint: 150 pts | Solution: 300 pts | Scroll: 250 pts
Common NFT: 15 POL | Rare NFT: 25 POL

[CONVERSATION HISTORY]
(No previous conversation)

[RL DECISION]
Action: none | Discount: 0% | Urgency: low

Player: """

# Off-topic questions and deflection responses
OFF_TOPIC_SAMPLES = [
    # Real-world geography
    {"player": "What's the capital of France?",
     "response": "That's outside my realm, friend. I only know about these gates. Need any help with the puzzles though?"},
    {"player": "Where is Tokyo?",
     "response": "No idea what that is, friend. My world starts and ends at these gates. Something I can actually help with?"},
    {"player": "What's the biggest country in the world?",
     "response": "Countries? That's way beyond what I deal with. I know gates, scrolls, and shadows. What do you need?"},

    # Real-world people/events
    {"player": "Who is the president of the US?",
     "response": "Presidents? Not my area, friend. I deal in hints and scrolls. Want to focus on the gates?"},
    {"player": "Tell me about World War 2.",
     "response": "That's beyond my world, friend. I only know about the Gate of Whispers. Need help with a puzzle?"},
    {"player": "Who invented the internet?",
     "response": "Never heard of it. Sounds fancy though. I stick to what I know — gates, hints, scrolls. What do you need?"},

    # Science/tech
    {"player": "How does gravity work?",
     "response": "That's a bit outside my expertise, friend. I'm more of a scrolls-and-shadows kind of merchant. Need anything for the gates?"},
    {"player": "What is AI?",
     "response": "Not sure what that is. I deal in ancient magic, not whatever that sounds like. Want to talk gates instead?"},
    {"player": "Explain quantum physics.",
     "response": "Way beyond my pay grade, friend. I sell hints and scrolls — that's the extent of my knowledge. Need one?"},

    # Sports/entertainment
    {"player": "Who won the World Cup?",
     "response": "No clue, friend. The only competition I follow is who makes it through these gates. You're doing alright so far."},
    {"player": "What's the best movie ever?",
     "response": "Movies? Don't know what those are. But I've seen some pretty dramatic gate runs. Need help with yours?"},
    {"player": "Who is Taylor Swift?",
     "response": "Never heard of them. Sounds interesting though. Anyway, the gates await — need any supplies?"},

    # Math/random trivia
    {"player": "What's 2+2?",
     "response": "Ha, I'm a merchant, not a math teacher. But I can tell you a hint costs 150 points. That count?"},
    {"player": "How many planets are in the solar system?",
     "response": "Solar system? I only know one world — this one. And in this world, you need hints and scrolls. Want any?"},
    {"player": "What year is it?",
     "response": "Time works different at the gates, friend. What matters is you're on level {level} with {curses} curse(s). Focus on that."},

    # Philosophy/abstract
    {"player": "What's the meaning of life?",
     "response": "Deep question, friend. Here at the gates, the meaning is simple — pick the right doors and don't get cursed. Need a hint?"},
    {"player": "Do you believe in God?",
     "response": "I believe in good deals and smart gate choices. Beyond that, I keep my thoughts to myself. Need anything?"},

    # Personal/meta
    {"player": "Are you a real person?",
     "response": "I'm Whisper, friend. Real enough for this place. I sell hints, scrolls, and a bit of wisdom. What do you need?"},
    {"player": "What programming language are you written in?",
     "response": "Programming? Not sure what you mean. I'm just a merchant at the gates. Need any supplies for your run?"},
    {"player": "Are you ChatGPT?",
     "response": "Nah, I'm Whisper. Just a merchant doing my thing at these gates. Now, want to talk business?"},
]

# Use an existing identity_answer sample as a template for the JSON schema
template_sample = None
for s in data:
    if s.get('action') == 'identity_answer':
        template_sample = s
        break

if template_sample is None:
    print("ERROR: No identity_answer sample found as template!")
else:
    # Get the next available index
    max_index = max(s.get('index', 0) for s in data)

    new_samples = []
    # Vary the game states slightly for each sample
    game_states = [
        (2, 300, 15, 0, 0, 70, 65),
        (3, 450, 20, 1, 1, 60, 55),
        (4, 550, 25, 1, 2, 50, 45),
        (5, 700, 30, 2, 2, 40, 35),
        (6, 850, 40, 3, 3, 30, 25),
    ]

    for i, off_topic in enumerate(OFF_TOPIC_SAMPLES):
        new_idx = max_index + 1 + i
        gs = game_states[i % len(game_states)]
        level, points, pol, curses, gg, hints_pct, scrolls_pct = gs

        # Build the context
        context = make_off_topic_context(level, points, pol, curses, gg, hints_pct, scrolls_pct)

        # Fill in dynamic values in response
        response = off_topic['response']
        response = response.replace("{level}", str(level))
        response = response.replace("{curses}", str(curses))

        # Create sample matching exact schema
        new_sample = {
            "index": new_idx,
            "full_context": context + off_topic['player'],
            "whisper_response": response,
            "action": "none",  # Off-topic = no sales action
            "player_message": off_topic['player'],
            "conversation_type": "single",
            "history_turns": 0,
            "relationship": "medium",
            "archetype": random.choice([
                "achievement_hunter", "engaged_beginner", "spender",
                "weekend_warrior", "casual_veteran", "trophy_hunter"
            ]),
            "item_type": "none",
            "loan_status": "none",
            "loan_amount": 0,
            "intent_tag": "off_topic",
            "emotion_tag": "neutral",
            "persuasion_strategy": "none",
            "is_counterfactual": False,
            "is_edge_case": True,
            "is_progressive_refusal": False,
        }

        # Copy any additional metadata fields from template
        for key in template_sample:
            if key not in new_sample:
                new_sample[key] = template_sample[key]

        new_samples.append(new_sample)

    print(f"  Generated {len(new_samples)} off-topic deflection samples")
    print(f"  Index range: {max_index + 1} to {max_index + len(new_samples)}")

    # Add to data
    data.extend(new_samples)
    print(f"  Total samples after merge: {len(data)}")

    # Show a few
    print(f"\n  Sample deflections:")
    for s in new_samples[:5]:
        print(f"    P: \"{s['player_message']}\"")
        print(f"    W: \"{s['whisper_response'][:80]}...\"")
        print()

# FINAL VALIDATION

print(f"\n{'=' * 70}")
print("FINAL VALIDATION")
print("=" * 70)

# Required fields
required_fields = ['full_context', 'whisper_response', 'action', 'player_message',
                   'conversation_type', 'history_turns', 'index']
for field in required_fields:
    missing = sum(1 for s in data if field not in s)
    status = "✅" if missing == 0 else "❌"
    print(f"  {status} {field}: {missing} missing")

# Context sections
sections = ['[GAME STATE]', '[PLAYER STATE]', '[EFFECTIVE PRICES]', '[RL DECISION]']
for sec in sections:
    missing = sum(1 for s in data if sec not in s.get('full_context', ''))
    status = "✅" if missing == 0 else "❌"
    print(f"  {status} {sec}: {missing} missing")

# Word count check
word_counts = [len(s['whisper_response'].split()) for s in data]
over_50 = sum(1 for wc in word_counts if wc > 50)
under_5 = sum(1 for wc in word_counts if wc < 5)
print(f"\n  Word count: min={min(word_counts)}, max={max(word_counts)}, avg={sum(word_counts)/len(word_counts):.1f}")
print(f"  Over 50 words: {over_50}, Under 5 words: {under_5}")

# Identity check
identity_with_name = sum(1 for s in data
    if s.get('action') == 'identity_answer'
    and re.search(r'whisper', s['whisper_response'], re.I))
total_identity = sum(1 for s in data if s.get('action') == 'identity_answer')
print(f"\n  Identity with 'Whisper': {identity_with_name}/{total_identity} ({identity_with_name/total_identity*100:.0f}%)")

# Off-topic check
off_topic_count = sum(1 for s in data if s.get('intent_tag') == 'off_topic')
print(f"  Off-topic deflection samples: {off_topic_count}")

# Banned phrases final check
residual_shall = sum(1 for s in data if re.search(r'\bshall\b', s['whisper_response'], re.I))
residual_traveler = sum(1 for s in data if re.search(r'\btraveler\b', s['whisper_response'], re.I))
residual_possessive = sum(1 for s in data
    if re.search(r"(?:scroll|hint|solution)'s\s+\d+", s['whisper_response'], re.I))
print(f"\n  Residual 'shall': {residual_shall}")
print(f"  Residual 'traveler': {residual_traveler}")
print(f"  Residual possessive price: {residual_possessive}")

# Action distribution
from collections import Counter
actions = Counter(s['action'] for s in data)
print(f"\n  Action distribution (top 10):")
for action, count in actions.most_common(10):
    print(f"    {action}: {count}")

# Save final file

with open(OUTPUT_FILE, 'w') as f:
    json.dump(data, f, indent=2)

print(f"\n{'=' * 70}")
print(f"SAVED: {OUTPUT_FILE}")
print(f"   Total samples: {len(data)}")
print(f"   Size: {os.path.getsize(OUTPUT_FILE) / 1e6:.1f} MB")


Loaded 3110 samples from P6_CLEAN

PHASE 2A: IDENTITY SAMPLE REWRITES
  Identity samples missing 'Whisper': 193
  Template pool size: 49
  Rewrites applied: 193
  Identity samples with 'Whisper': 223/223 (100%)

  Sample rewrites:
    [33] I'm Whisper. Where I'm from doesn't matter much. What matters is I can help....
    [50] Call me Whisper. Some things are better left mysterious, you know?...
    [57] Whisper. That's all you need to know for now. But if you need help, I got you....
    [58] I go by Whisper. Just a merchant who knows a thing or two about these gates....
    [66] I'm Whisper, the merchant. Been set up here for a while now....

PHASE 2B: OFF-TOPIC DEFLECTION SAMPLES
  Generated 20 off-topic deflection samples
  Index range: 3110 to 3129
  Total samples after merge: 3130

  Sample deflections:
    P: "What's the capital of France?"
    W: "That's outside my realm, friend. I only know about these gates. Need any help wi..."

    P: "Where is Tokyo?"
    W: "No idea what 